# Build Two Multitask Datasets
Creates two datasets:
1. **Fully labeled dataset**: All classification images have masks, all segmentation images have labels
2. **Partially labeled dataset**: Half of each subset has both labels and masks, half has only one

In [7]:
# Common imports and path configuration
from pathlib import Path
import random
import shutil
import math

import pandas as pd

random.seed(42)
project_root = Path("/Volumes/T7/largeProject_copy")
classification_root = project_root / "dataset" / "classification"
segmentation_root = project_root / "dataset" / "segmentation"
segmentation_csv = project_root / "seg_labels.csv"
generated_masks_root = project_root / "generated_masks"

# Output directories for both datasets
output_fully_labeled = project_root / "multitask_fully_labeled"
output_partially_labeled = project_root / "multitask_partially_labeled"

splits = ("train", "val", "test")
class_columns = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]

print(f"Classification root: {classification_root}")
print(f"Segmentation root: {segmentation_root}")
print(f"Generated masks root: {generated_masks_root}")
print(f"Segmentation CSV: {segmentation_csv}")

Classification root: /Volumes/T7/largeProject_copy/dataset/classification
Segmentation root: /Volumes/T7/largeProject_copy/dataset/segmentation
Generated masks root: /Volumes/T7/largeProject_copy/generated_masks
Segmentation CSV: /Volumes/T7/largeProject_copy/seg_labels.csv


In [8]:
# Gather all classification images with labels and check for generated masks
if not segmentation_csv.exists():
    raise FileNotFoundError(f"Segmentation CSV not found at {segmentation_csv}")

classification_records = []
classification_ids = set()

for split in splits:
    gt_dir = classification_root / split / "ground_truth"
    input_dir = classification_root / split / "input"
    csv_files = sorted(gt_dir.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No classification label CSV found under {gt_dir}")
    cls_df = pd.read_csv(csv_files[0])
    
    for row in cls_df.itertuples(index=False):
        image_id = str(row.image)
        image_path = input_dir / f"{image_id}.jpg"
        if not image_path.exists():
            continue
        if image_id in classification_ids:
            continue
            
        # Check if generated mask exists
        generated_mask_path = generated_masks_root / split / f"{image_id}_segmentation.png"
        has_mask = generated_mask_path.exists()
        
        labels = [int(float(getattr(row, col))) for col in class_columns]
        record = {
            "image_id": image_id,
            "image_path": image_path,
            "mask_path": generated_mask_path if has_mask else None,
            "has_mask": has_mask,
            "split": split,
            "labels": labels,
        }
        classification_records.append(record)
        classification_ids.add(image_id)

# Gather segmentation images with masks and check for labels
seg_entries_by_id = {}
seg_df = pd.read_csv(segmentation_csv)
missing_class_cols = [col for col in class_columns if col not in seg_df.columns]
if missing_class_cols:
    raise RuntimeError(
        "Segmentation labels CSV does not contain required classification columns: "
        + ", ".join(missing_class_cols),
    )

# First pass: segmentation images with labels
for row in seg_df.itertuples(index=False):
    rel_path = Path(str(row.image))
    split = rel_path.parts[0]
    image_path = segmentation_root / rel_path
    mask_path = segmentation_root / split / "ground_truth" / f"{rel_path.stem}_segmentation.png"
    if not image_path.exists() or not mask_path.exists():
        continue
    image_id = rel_path.stem
    labels = [int(float(getattr(row, col))) for col in class_columns]
    seg_entries_by_id[image_id] = {
        "image_id": image_id,
        "image_path": image_path,
        "mask_path": mask_path,
        "split": split,
        "labels": labels,
        "has_labels": True,
    }

# Second pass: segmentation images without labels
for split in splits:
    input_dir = segmentation_root / split / "input"
    mask_dir = segmentation_root / split / "ground_truth"
    for image_path in sorted(input_dir.glob("*.jpg")):
        image_id = image_path.stem
        mask_path = mask_dir / f"{image_id}_segmentation.png"
        if not mask_path.exists():
            continue
        if image_id in seg_entries_by_id:
            continue
        seg_entries_by_id[image_id] = {
            "image_id": image_id,
            "image_path": image_path,
            "mask_path": mask_path,
            "split": split,
            "labels": None,
            "has_labels": False,
        }

segmentation_entries = list(seg_entries_by_id.values())

print(f"\n📊 Data Collection Summary:")
print(f"Classification images: {len(classification_records)}")
print(f"  - With generated masks: {sum(1 for r in classification_records if r['has_mask'])}")
print(f"  - Without masks: {sum(1 for r in classification_records if not r['has_mask'])}")
print(f"Segmentation images: {len(segmentation_entries)}")
print(f"  - With labels: {sum(1 for e in segmentation_entries if e['has_labels'])}")
print(f"  - Without labels: {sum(1 for e in segmentation_entries if not e['has_labels'])}")


📊 Data Collection Summary:
Classification images: 11720
  - With generated masks: 11720
  - Without masks: 0
Segmentation images: 3694
  - With labels: 3694
  - Without labels: 0


In [9]:
# Build Fully Labeled Dataset
# All classification images with generated masks + All segmentation images with labels
print("\n🔨 Building Fully Labeled Dataset...")

fully_labeled_records = []

# Add classification images that have generated masks
for record in classification_records:
    if not record["has_mask"]:
        continue  # Skip images without masks
    fully_labeled_records.append({
        "image_id": record["image_id"],
        "image_path": record["image_path"],
        "mask_path": record["mask_path"],
        "split": record["split"],
        "labels": record["labels"],
        "source": "classification"
    })

# Add segmentation images that have labels
for entry in segmentation_entries:
    if not entry["has_labels"]:
        continue  # Skip images without labels
    fully_labeled_records.append({
        "image_id": entry["image_id"],
        "image_path": entry["image_path"],
        "mask_path": entry["mask_path"],
        "split": entry["split"],
        "labels": entry["labels"],
        "source": "segmentation"
    })

# Create directory structure and copy files
output_fully_labeled.mkdir(parents=True, exist_ok=True)
(output_fully_labeled / "images").mkdir(exist_ok=True)
(output_fully_labeled / "masks").mkdir(exist_ok=True)

manifest_rows = []
for record in fully_labeled_records:
    # Copy image
    dst_image = output_fully_labeled / "images" / f"{record['image_id']}.jpg"
    if not dst_image.exists():
        shutil.copy2(record["image_path"], dst_image)
    
    # Copy mask
    dst_mask = output_fully_labeled / "masks" / f"{record['image_id']}_segmentation.png"
    if not dst_mask.exists():
        shutil.copy2(record["mask_path"], dst_mask)
    
    # Build manifest row
    manifest_row = {
        "image_id": record["image_id"],
        "image": f"images/{record['image_id']}.jpg",
        "mask": f"masks/{record['image_id']}_segmentation.png",
        "split": record["split"],
        "source": record["source"]
    }
    for i, col in enumerate(class_columns):
        manifest_row[col] = record["labels"][i]
    manifest_rows.append(manifest_row)

# Save manifest
manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(output_fully_labeled / "manifest.csv", index=False)

print(f"✅ Fully Labeled Dataset Created:")
print(f"   Total samples: {len(manifest_rows)}")
print(f"   From classification: {sum(1 for r in manifest_rows if r['source'] == 'classification')}")
print(f"   From segmentation: {sum(1 for r in manifest_rows if r['source'] == 'segmentation')}")
print(f"   Train: {sum(1 for r in manifest_rows if r['split'] == 'train')}")
print(f"   Val: {sum(1 for r in manifest_rows if r['split'] == 'val')}")
print(f"   Test: {sum(1 for r in manifest_rows if r['split'] == 'test')}")


🔨 Building Fully Labeled Dataset...
✅ Fully Labeled Dataset Created:
   Total samples: 15414
   From classification: 11720
   From segmentation: 3694
   Train: 12609
   Val: 293
   Test: 2512


In [10]:
# Build Partially Labeled Dataset
# 50% of classification images have both (labels + masks), 50% have only labels
# 50% of segmentation images have both (masks + labels), 50% have only masks
print("\n🔨 Building Partially Labeled Dataset...")

# Separate classification images with and without masks
cls_with_masks = [r for r in classification_records if r["has_mask"]]
cls_without_masks = [r for r in classification_records if not r["has_mask"]]

# Shuffle and split classification images with masks 50/50
random.shuffle(cls_with_masks)
n_cls_paired = len(cls_with_masks) // 2
cls_paired = cls_with_masks[:n_cls_paired]
cls_label_only = cls_with_masks[n_cls_paired:] + cls_without_masks  # Rest + those without masks

# Separate segmentation images with and without labels
seg_with_labels = [e for e in segmentation_entries if e["has_labels"]]
seg_without_labels = [e for e in segmentation_entries if not e["has_labels"]]

# Shuffle and split segmentation images with labels 50/50
random.shuffle(seg_with_labels)
n_seg_paired = len(seg_with_labels) // 2
seg_paired = seg_with_labels[:n_seg_paired]
seg_mask_only = seg_with_labels[n_seg_paired:] + seg_without_labels  # Rest + those without labels

# Create directory structure
output_partially_labeled.mkdir(parents=True, exist_ok=True)
(output_partially_labeled / "paired" / "images").mkdir(parents=True, exist_ok=True)
(output_partially_labeled / "paired" / "masks").mkdir(parents=True, exist_ok=True)
(output_partially_labeled / "classification_only" / "images").mkdir(parents=True, exist_ok=True)
(output_partially_labeled / "segmentation_only" / "images").mkdir(parents=True, exist_ok=True)
(output_partially_labeled / "segmentation_only" / "masks").mkdir(parents=True, exist_ok=True)

# Process paired samples (both modalities)
paired_manifest = []
for record in cls_paired:
    dst_image = output_partially_labeled / "paired" / "images" / f"{record['image_id']}.jpg"
    dst_mask = output_partially_labeled / "paired" / "masks" / f"{record['image_id']}_segmentation.png"
    if not dst_image.exists():
        shutil.copy2(record["image_path"], dst_image)
    if not dst_mask.exists():
        shutil.copy2(record["mask_path"], dst_mask)
    
    row = {
        "image_id": record["image_id"],
        "image": f"paired/images/{record['image_id']}.jpg",
        "mask": f"paired/masks/{record['image_id']}_segmentation.png",
        "split": record["split"],
        "source": "classification"
    }
    for i, col in enumerate(class_columns):
        row[col] = record["labels"][i]
    paired_manifest.append(row)

for entry in seg_paired:
    dst_image = output_partially_labeled / "paired" / "images" / f"{entry['image_id']}.jpg"
    dst_mask = output_partially_labeled / "paired" / "masks" / f"{entry['image_id']}_segmentation.png"
    if not dst_image.exists():
        shutil.copy2(entry["image_path"], dst_image)
    if not dst_mask.exists():
        shutil.copy2(entry["mask_path"], dst_mask)
    
    row = {
        "image_id": entry["image_id"],
        "image": f"paired/images/{entry['image_id']}.jpg",
        "mask": f"paired/masks/{entry['image_id']}_segmentation.png",
        "split": entry["split"],
        "source": "segmentation"
    }
    for i, col in enumerate(class_columns):
        row[col] = entry["labels"][i]
    paired_manifest.append(row)

# Process classification-only samples
cls_only_manifest = []
for record in cls_label_only:
    dst_image = output_partially_labeled / "classification_only" / "images" / f"{record['image_id']}.jpg"
    if not dst_image.exists():
        shutil.copy2(record["image_path"], dst_image)
    
    row = {
        "image_id": record["image_id"],
        "image": f"classification_only/images/{record['image_id']}.jpg",
        "mask": None,
        "split": record["split"],
        "source": "classification"
    }
    for i, col in enumerate(class_columns):
        row[col] = record["labels"][i]
    cls_only_manifest.append(row)

# Process segmentation-only samples
seg_only_manifest = []
for entry in seg_mask_only:
    dst_image = output_partially_labeled / "segmentation_only" / "images" / f"{entry['image_id']}.jpg"
    dst_mask = output_partially_labeled / "segmentation_only" / "masks" / f"{entry['image_id']}_segmentation.png"
    if not dst_image.exists():
        shutil.copy2(entry["image_path"], dst_image)
    if not dst_mask.exists():
        shutil.copy2(entry["mask_path"], dst_mask)
    
    row = {
        "image_id": entry["image_id"],
        "image": f"segmentation_only/images/{entry['image_id']}.jpg",
        "mask": f"segmentation_only/masks/{entry['image_id']}_segmentation.png",
        "split": entry["split"],
        "source": "segmentation"
    }
    for i, col in enumerate(class_columns):
        row[col] = None if entry["labels"] is None else entry["labels"][i]
    seg_only_manifest.append(row)

# Save manifests
pd.DataFrame(paired_manifest).to_csv(output_partially_labeled / "paired" / "manifest.csv", index=False)
pd.DataFrame(cls_only_manifest).to_csv(output_partially_labeled / "classification_only" / "manifest.csv", index=False)
pd.DataFrame(seg_only_manifest).to_csv(output_partially_labeled / "segmentation_only" / "manifest.csv", index=False)

print(f"✅ Partially Labeled Dataset Created:")
print(f"   Paired: {len(paired_manifest)}")
print(f"     - From classification: {sum(1 for r in paired_manifest if r['source'] == 'classification')}")
print(f"     - From segmentation: {sum(1 for r in paired_manifest if r['source'] == 'segmentation')}")
print(f"   Classification-only: {len(cls_only_manifest)}")
print(f"   Segmentation-only: {len(seg_only_manifest)}")
print(f"   Total: {len(paired_manifest) + len(cls_only_manifest) + len(seg_only_manifest)}")


🔨 Building Partially Labeled Dataset...
✅ Partially Labeled Dataset Created:
   Paired: 7707
     - From classification: 5860
     - From segmentation: 1847
   Classification-only: 5860
   Segmentation-only: 1847
   Total: 15414


## Dataset Creation Complete
Created two datasets for multi-task learning experiments:
1. **Fully labeled**: All samples have both classification labels and segmentation masks
2. **Partially labeled**: 50% paired (both modalities), 50% single modality only